In [ ]:
#step-1-configuration
!pip install groq --q
print("libraries install successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.6 MB/s eta 0:00:00
libraries install successfully!


In [ ]:
#step-2-import
import sqlite3
import pandas as pd
import os
from groq import Groq
import re
print("import successfully!")

import successfully!


In [ ]:
#initialize the ai model
import os
os.environ["GROQ_API_KEY"]="groq_api_key"
client=Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL="llama-3.1-8b-instant"
print("Groq client initialised successfully!")
print(f"using model:{MODEL}")

Groq client initialised successfully!
using model:llama-3.1-8b-instant


In [ ]:
# import csv files
import io
csv_data = """student_id,name,age,gender,subject,marks,attendance,grade
1,Aarav Sharma,20,Male,Mathematics,88,92,A
2,Priya Patel,21,Female,Science,76,85,B
3,Rohan Mehta,20,Male,Programming,95,98,A+
4,Sneha Iyer,22,Female,Mathematics,62,78,C
5,Arjun Nair,21,Male,Programming,91,94,A+
6,Divya Krishnan,20,Female,Science,83,88,A
7,Karan Singh,22,Male,Mathematics,74,81,B
8,Ananya Gupta,21,Female,Programming,89,96,A
9,Vikram Reddy,20,Male,Science,70,79,B
10,Pooja Sharma,22,Female,Mathematics,55,72,D
11,Aditya Kumar,21,Male,Programming,97,99,A+
12,Meera Nambiar,20,Female,Science,81,87,A
13,Rahul Desai,22,Male,Mathematics,68,80,C
14,Kavitha Rajan,21,Female,Programming,86,93,A
15,Nikhil Verma,20,Male,Science,77,84,B
16,Swathi Pillai,22,Female,Mathematics,90,95,A+
17,Manish Joshi,21,Male,Programming,73,82,B
18,Lavanya Menon,20,Female,Science,66,76,C
19,Suresh Babu,22,Male,Mathematics,82,89,A
20,Anjali Singh,21,Female,Programming,94,97,A+
21,Deepak Nair,20,Male,Science,79,86,B
22,Rekha Sharma,22,Female,Mathematics,58,73,D
23,Sanjay Patel,21,Male,Programming,88,91,A
24,Usha Iyer,20,Female,Science,84,90,A
25,Vijay Kumar,22,Male,Mathematics,71,83,B
26,Nandita Rao,21,Female,Programming,92,96,A+
27,Ashok Reddy,20,Male,Science,65,77,C
28,Sunita Gupta,22,Female,Mathematics,87,93,A
29,Ravi Krishnan,21,Male,Programming,78,88,B
30,Bhavna Mehta,20,Female,Science,93,98,A+"""

df=pd.read_csv(io.StringIO(csv_data))
print(f"dataset loaded: {len(df)} rows, {len(df.columns)} columns")
df.head()

dataset loaded: 30 rows, 8 columns


,student_id,name,age,gender,subject,marks,attendance,grade
0,1,Aarav Sharma,20,Male,Mathematics,88,92,A
1,2,Priya Patel,21,Female,Science,76,85,B
2,3,Rohan Mehta,20,Male,Programming,95,98,A+
3,4,Sneha Iyer,22,Female,Mathematics,62,78,C
4,5,Arjun Nair,21,Male,Programming,91,94,A+


In [ ]:
#connection
conn=sqlite3.connect("college.db")
df.to_sql("students",conn,if_exists="replace",index=False)
print("database created:college.db")
print("table,'students' created with 30 student records")
test_df=pd.read_sql_query("select count(*) as total_rows from students", con=conn)
print(f"\nverification:{test_df['total_rows'][0]} rows in database")

database created:college.db
table,'students' created with 30 student records

verification:30 rows in database


In [ ]:
def get_schema(conn, table_name="students"):
  cursor = conn.cursor()
  cursor.execute(f"PRAGMA table_info({table_name})")
  columns = cursor.fetchall()
  schema_lines = [f"Table: {table_name}"]
  schema_lines.append("Columns:")
  for col in columns:
    schema_lines.append(f"\t{col[1]} ({col[2]})")
  return "\n".join(schema_lines)
print(get_schema(conn))
sample = get_schema(conn)
sample_data = pd.read_sql_query("SELECT * FROM students LIMIT 3", conn)
print("Sample data from 'students' table:")
display(sample_data)

Table: students
Columns:
	student_id (INTEGER)
	name (TEXT)
	age (INTEGER)
	gender (TEXT)
	subject (TEXT)
	marks (INTEGER)
	attendance (INTEGER)
	grade (TEXT)
Sample data from 'students' table:


,student_id,name,age,gender,subject,marks,attendance,grade
0,1,Aarav Sharma,20,Male,Mathematics,88,92,A
1,2,Priya Patel,21,Female,Science,76,85,B
2,3,Rohan Mehta,20,Male,Programming,95,98,A+


In [ ]:
def generate_sql(user_question,schema_text,client,model):
  system_prompt = f"""You are an expert SQL assistant.
You are connected to a SQLite database with the following structure:


{schema_text}


Rules you must follow:
1. Generate ONLY a valid SQLite SQL query.
2. Do not include any explanation or text — only the SQL query.
3. Do not use markdown code blocks. Return the raw SQL only.
4. The table name is: students
5. Only use column names that exist in the schema above.
6. Use single quotes for string values in WHERE clauses (example: WHERE subject = 'Programming').
7. If the user asks for top N, use ORDER BY marks DESC LIMIT N.
"""
  response=client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_question}
    ],
    temperature=0.0
  )
  sql_query=response.choices[0].message.content.strip()
  return sql_query
question="show me first 5 female students"
print(f"question:{question}")
print("\ngenerating SQL...")
sql=generate_sql(question,sample,client,MODEL)
print(f"\nGenerated SQL:\n{sql}")

question:show me first 5 female students

generating SQL...

Generated SQL:
SELECT * FROM students WHERE gender = 'Female' ORDER BY marks DESC LIMIT 5


In [ ]:
def execute_sql(sql_query,conn):
   clean_sql=sql_query.strip()
   clean_sql=re.sub(r'```sql\s*','',clean_sql)
   clean_sql=re.sub(r'```\s*','',clean_sql)
   clean_sql=clean_sql.strip()
   try:
    result_df=pd.read_sql_query(clean_sql,conn)
    return result_df,None
   except Exception as e:

    return None,str(e)
print(f"executing sql:{sql}")
result,error=execute_sql(sql,conn)
if error:
  print(f"Error:{error}")
else:
  print(f"\nquery result:\n{sql}rows:" )
print(result)

executing sql:SELECT * FROM students WHERE gender = 'Female' ORDER BY marks DESC LIMIT 5

query result:
SELECT * FROM students WHERE gender = 'Female' ORDER BY marks DESC LIMIT 5rows:
   student_id           name  age  gender      subject  marks  attendance  \
0          20   Anjali Singh   21  Female  Programming     94          97   
1          30   Bhavna Mehta   20  Female      Science     93          98   
2          26    Nandita Rao   21  Female  Programming     92          96   
3          16  Swathi Pillai   22  Female  Mathematics     90          95   
4           8   Ananya Gupta   21  Female  Programming     89          96   

  grade  
0    A+  
1    A+  
2    A+  
3    A+  
4     A  


In [ ]:
def text_to_sql_agent(user_question,conn,client,model,verbose=True):
  print(f"user question:{user_question}")
  if verbose:
    print("\n[step1] Reading database schema...")
  schema_text=get_schema(conn)
  if verbose:
    print("schema loaded successfully")
  if verbose:
    print("\n[step 2] generating SQL query with Groq LLM...")
  generated_sql=generate_sql(user_question,schema_text,client,model)
  if verbose:
    print(f"Generated SQL:\n{generated_sql}")
  if verbose:
    print("\n[step 3] executing generated SQL...")
  result_df,error=execute_sql(generated_sql,conn)
  if error:
    print(f"SQL Execution Error:{error}")
    return None,generated_sql

  if verbose:
    print(f"\nQuery result for SQL:\n{generated_sql}")
    display(result_df)
  return result_df,generated_sql
  if verbose:
    print(f"\n[step 4] query returned{len(result_df)},row(s)")
    print("\nresult:")
    display(result_df.to_string(index=False))
  return result_df,generated_sql
result,sql_used=text_to_sql_agent(
    "show top 5 students in programming",
    conn,client,MODEL
)

user question:show top 5 students in programming

[step1] Reading database schema...
schema loaded successfully

[step 2] generating SQL query with Groq LLM...
Generated SQL:
SELECT name FROM students WHERE subject = 'Programming' ORDER BY marks DESC LIMIT 5

[step 3] executing generated SQL...

Query result for SQL:
SELECT name FROM students WHERE subject = 'Programming' ORDER BY marks DESC LIMIT 5


,name
0,Aditya Kumar
1,Rohan Mehta
2,Anjali Singh
3,Nandita Rao
4,Arjun Nair


In [ ]:
result1, _ =text_to_sql_agent(
    "show me all students who study Mathematics",
    conn,client,MODEL
)

user question:show me all students who study Mathematics

[step1] Reading database schema...
schema loaded successfully

[step 2] generating SQL query with Groq LLM...
Generated SQL:
SELECT * FROM students WHERE subject = 'Mathematics'

[step 3] executing generated SQL...

Query result for SQL:
SELECT * FROM students WHERE subject = 'Mathematics'


,student_id,name,age,gender,subject,marks,attendance,grade
0,1,Aarav Sharma,20,Male,Mathematics,88,92,A
1,4,Sneha Iyer,22,Female,Mathematics,62,78,C
2,7,Karan Singh,22,Male,Mathematics,74,81,B
3,10,Pooja Sharma,22,Female,Mathematics,55,72,D
4,13,Rahul Desai,22,Male,Mathematics,68,80,C
5,16,Swathi Pillai,22,Female,Mathematics,90,95,A+
6,19,Suresh Babu,22,Male,Mathematics,82,89,A
7,22,Rekha Sharma,22,Female,Mathematics,58,73,D
8,25,Vijay Kumar,22,Male,Mathematics,71,83,B
9,28,Sunita Gupta,22,Female,Mathematics,87,93,A


In [ ]:
result1,_=text_to_sql_agent(
    "show me all students who study Mathematics",
    conn,client,MODEL
)

user question:show me all students who study Mathematics

[step1] Reading database schema...
schema loaded successfully

[step 2] generating SQL query with Groq LLM...
Generated SQL:
SELECT * FROM students WHERE subject = 'Mathematics'

[step 3] executing generated SQL...

Query result for SQL:
SELECT * FROM students WHERE subject = 'Mathematics'


,student_id,name,age,gender,subject,marks,attendance,grade
0,1,Aarav Sharma,20,Male,Mathematics,88,92,A
1,4,Sneha Iyer,22,Female,Mathematics,62,78,C
2,7,Karan Singh,22,Male,Mathematics,74,81,B
3,10,Pooja Sharma,22,Female,Mathematics,55,72,D
4,13,Rahul Desai,22,Male,Mathematics,68,80,C
5,16,Swathi Pillai,22,Female,Mathematics,90,95,A+
6,19,Suresh Babu,22,Male,Mathematics,82,89,A
7,22,Rekha Sharma,22,Female,Mathematics,58,73,D
8,25,Vijay Kumar,22,Male,Mathematics,71,83,B
9,28,Sunita Gupta,22,Female,Mathematics,87,93,A


In [ ]:
result1,_=text_to_sql_agent(
    "show me top 5students who study Mathematics where filtering by age",
    conn,client,MODEL
)

user question:show me top 5students who study Mathematics where filtering by age

[step1] Reading database schema...
schema loaded successfully

[step 2] generating SQL query with Groq LLM...
Generated SQL:
SELECT name FROM students WHERE subject = 'Mathematics' AND age > 18 ORDER BY marks DESC LIMIT 5

[step 3] executing generated SQL...

Query result for SQL:
SELECT name FROM students WHERE subject = 'Mathematics' AND age > 18 ORDER BY marks DESC LIMIT 5


,name
0,Swathi Pillai
1,Aarav Sharma
2,Sunita Gupta
3,Suresh Babu
4,Karan Singh


In [ ]:
result1,_=text_to_sql_agent(
    "count of all students in maths and their marks",
    conn,client,MODEL
)

user question:count of all students in maths and their marks

[step1] Reading database schema...
schema loaded successfully

[step 2] generating SQL query with Groq LLM...
Generated SQL:
SELECT COUNT(student_id), marks FROM students WHERE subject = 'Maths'

[step 3] executing generated SQL...

Query result for SQL:
SELECT COUNT(student_id), marks FROM students WHERE subject = 'Maths'


,COUNT(student_id),marks
0,0,None


In [ ]:
result1,_=text_to_sql_agent(
    "who all got the A+ grade --- need to show the students name",
    conn,client,MODEL
)

user question:who all got the A+ grade --- need to show the students name

[step1] Reading database schema...
schema loaded successfully

[step 2] generating SQL query with Groq LLM...
Generated SQL:
SELECT name FROM students WHERE grade = 'A+'

[step 3] executing generated SQL...

Query result for SQL:
SELECT name FROM students WHERE grade = 'A+'


,name
0,Rohan Mehta
1,Arjun Nair
2,Aditya Kumar
3,Swathi Pillai
4,Anjali Singh
5,Nandita Rao
6,Bhavna Mehta


In [ ]:
result1,_=text_to_sql_agent(
    "top 5 students oda name kudu athuvu yarukula  age 18 ah eruko  kudu",
    conn,client,MODEL
)

user question:top 5 students oda name kudu athuvu yarukula  age 18 ah eruko  kudu

[step1] Reading database schema...
schema loaded successfully

[step 2] generating SQL query with Groq LLM...
Generated SQL:
SELECT name FROM students WHERE age = 18 ORDER BY marks DESC LIMIT 5

[step 3] executing generated SQL...

Query result for SQL:
SELECT name FROM students WHERE age = 18 ORDER BY marks DESC LIMIT 5


,name
